In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, DoubleType, DateType, DataType
from pyspark.sql.functions import *
import re
import pandas as pd
import pyspark.pandas as pd
import unicodedata
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
from delta.tables import DeltaTable

Tried to attach usage logger `pyspark.databricks.pandas.usage_logger`, but an exception was raised: JVM wasn't initialised. Did you call it on executor side?


In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver_olist;

USE silver_olist;

In [0]:
# Banco Realizado na camada Bronze
banco_bronze = "olist"

#Lista dinamica de todas as tabelas da Bronze
tabelas_bronze = spark.catalog.listTables(banco_bronze)

#lista apenas com os nomes delas
lista_de_tabelas = [t.name for t in tabelas_bronze if t.name.startswith("lnd")]

print(lista_de_tabelas)

['lnd_bronze_customers', 'lnd_bronze_geolocation', 'lnd_bronze_order_items', 'lnd_bronze_order_payments', 'lnd_bronze_order_reviews', 'lnd_bronze_orders', 'lnd_bronze_product_category_name_translation', 'lnd_bronze_products', 'lnd_bronze_sellers']


In [0]:
#Verificando o Schema das Tabelas
for nome_tabela in lista_de_tabelas:
    print("\n" + "="*50)
    print(f" SCHEMA DA TABELA: {nome_tabela}")
    print("="*50)
    # Lendo o DataFrame atualizado até a fase anterior
    df_atual = spark.read.table(f"{banco_bronze}.{nome_tabela}") 
    
    # Exibe a estrutura de colunas e tipos
    df_atual.printSchema()


 SCHEMA DA TABELA: lnd_bronze_customers
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: long (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)


 SCHEMA DA TABELA: lnd_bronze_geolocation
root
 |-- geolocation_zip_code_prefix: long (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)


 SCHEMA DA TABELA: lnd_bronze_order_items
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)


 SCHEMA DA TABELA: lnd_bronze_order_payments
root
 |-- order_id: stri

In [0]:
banco_bronze = "olist"
nome_banco_silver = "silver_olist"  

# Dicionário de configurações para a deduplicação
config_cdc = {
    "lnd_bronze_orders": {"pk": "order_id", "sort_col": "order_purchase_timestamp"},
    "lnd_bronze_customers": {"pk": "customer_id", "sort_col": "customer_id"}, 
    "lnd_bronze_geolocation": {"pk": "geolocation_zip_code_prefix", "sort_col": "geolocation_zip_code_prefix"},
    "lnd_bronze_order_items": {"pk": "order_id", "sort_col": "shipping_limit_date"}, 
    "lnd_bronze_order_payments": {"pk": "order_id", "sort_col": "order_id"},
    "lnd_bronze_order_reviews": {"pk": "review_id", "sort_col": "review_creation_date"},
    "lnd_bronze_products": {"pk": "product_id", "sort_col": "product_id"},
    "lnd_bronze_sellers": {"pk": "seller_id", "sort_col": "seller_id"},
    "lnd_bronze_product_category_name_translation": {"pk": "product_category_name", "sort_col": "product_category_name"}
}

# Lista dinâmica buscando os dados reais da Bronze
tabelas_bronze = spark.catalog.listTables(banco_bronze)
lista_de_tabelas = [t.name for t in tabelas_bronze if t.name.startswith("lnd")]

print(f"Tabelas encontradas na Bronze para processar: {lista_de_tabelas}\n")


for nome_tabela in lista_de_tabelas:
    print("\n" + "="*60)
    print(f"INICIANDO PIPELINE DA TABELA: {nome_tabela}")
    print("="*60)
    
    # --- FASE 0: LEITURA DINÂMICA ---
    df_raw = spark.read.table(f"{banco_bronze}.{nome_tabela}")
    
    # --- FASE 1: SUA PADRONIZAÇÃO DE COLUNAS (SNAKE_CASE) ---
    # Aplica a função que limpa caracteres e padroniza os nomes
    mapeamento_colunas = {coluna: padronizar_nome_coluna(coluna) for coluna in df_raw.columns}
    df_padronizado = df_raw.withColumnsRenamed(mapeamento_colunas)
    print(f"   Padronizado: {nome_tabela} colunas {df_padronizado.columns}...")
    # - Tratar nulos
    mapeamento_chaves = {
        "lnd_bronze_customers":"customer_id",
        "lnd_bronze_order_items":"order_id",
        "lnd_bronze_order_payments" : "order_id",
        "lnd_bronze_order_reviews": "order_id",
        "lnd_bronze_orders"  : "order_id",
        "lnd_bronze_products": "product_id",
        "lnd_bronze_sellers": "seller_id"
    }
    chave_primaria = mapeamento_chaves.get(nome_tabela)
    if chave_primaria:
        # Remove linhas onde a chave primária é nula (Nulo de Erro)
        df_sem_nulos_graves = df_padronizado.dropna(subset=[chave_primaria])
    else:
        df_sem_nulos_graves = df_padronizado
    
    # --- FASE 2: SEU TRATAMENTO DE NULOS DINÂMICO ---
    # Remove linhas 100% vazias
    if nome_tabela == "olist_products_dataset":
        df_final_nulos = df_sem_nulos_graves.fillna({"product_category_name": "nao_informado"})
    # Se for a tabela de avaliações (reviews), podemos preencher comentários vazios
    elif nome_tabela == "olist_order_reviews_dataset":
        df_final_nulos = df_sem_nulos_graves.fillna({
            "review_comment_title": "sem_titulo",
            "review_comment_message": "sem_mensagem"
        })
    else:
        # Para as outras tabelas (como as datas de entrega em orders), mantemos os nulos de negócio como None
        df_final_nulos = df_sem_nulos_graves
    
    
    # --- FASE 3: CONVERSÃO DE TIPOS (CASTING) ---
    regras_de_casting = {
    "lnd_bronze_orders": {
        "order_purchase_timestamp": "timestamp",
        "order_approved_at": "timestamp",
        "order_delivered_carrier_date": "timestamp",
        "order_delivered_customer_date": "timestamp",
        "order_estimated_delivery_date": "timestamp"
    },
    "lnd_bronze_order_items": {
        "shipping_limit_date": "timestamp",
        "price": "double",
        "freight_value": "double"
    },
    "lnd_bronze_order_reviews": {
        "review_score": "integer",
        "review_creation_date": "timestamp",
        "review_answer_timestamp": "timestamp"
    },
    "lnd_bronze_products": {
        "product_photos_qty": "integer",
    },
    # Para os CEPs, passamos para string para não perder o zero à esquerda
    "lnd_bronze_customers": {
        "customer_zip_code_prefix": "string"
    },
    "lnd_bronze_sellers": {
        "seller_zip_code_prefix": "string"
    },
    "lnd_bronze_geolocation": {
        "geolocation_zip_code_prefix": "string"
    }
}
    
    df_atual = df_final_nulos 
    regras_da_tabela = regras_de_casting.get(nome_tabela, {})
    
    for coluna_nome, novo_tipo in regras_da_tabela.items():
        if novo_tipo == "integer":
            # Use try_cast for integer columns to handle invalid data gracefully
            df_atual = df_atual.withColumn(coluna_nome, expr(f"try_cast({coluna_nome} as int)"))
        elif novo_tipo == "timestamp":
            # Use try_cast for timestamp columns to handle invalid data gracefully
            df_atual = df_atual.withColumn(coluna_nome, expr(f"try_cast({coluna_nome} as timestamp)"))
        else:
            df_atual = df_atual.withColumn(coluna_nome, col(coluna_nome).cast(novo_tipo))

    # --- FASE 4: DEDUPLICAÇÃO DA CARGA ---
    config_atual = config_cdc.get(nome_tabela)
    pk_coluna = config_atual["pk"]
    sort_coluna = config_atual["sort_col"]

    print(f"Deduplicando os dados usando a chave primária: {pk_coluna}")

    window_spec = Window.partitionBy(pk_coluna).orderBy(col(sort_coluna).desc())

    df_deduplicado = df_atual.withColumn("rn", row_number().over(window_spec)) \
                             .filter(col("rn") == 1) \
                             .drop("rn")

    # --- FASE 5: CDC / MERGE NA TABELA SILVER ---
    # Altere para "stg_" se quiser o prefixo de stage, ou mantenha como está
    nome_tabela_silver = nome_tabela.replace("lnd_bronze_", "stg_") 
    tabela_destino = f"{nome_banco_silver}.{nome_tabela_silver}"

    tabela_existe = spark.catalog.tableExists(tabela_destino)

    if not tabela_existe:
        print(f"Tabela {tabela_destino} não encontrada. Criando carga inicial...")
        df_deduplicado.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(tabela_destino)
        print(f"Carga inicial criada com sucesso!")
    else:
        print(f"Tabela {tabela_destino} detectada. Executando MERGE incremental (CDC)...")
        
        delta_target = DeltaTable.forName(spark, tabela_destino)
        
        delta_target.alias("silver") \
            .merge(
                df_deduplicado.alias("bronze"),
                f"silver.{pk_coluna} = bronze.{pk_coluna}"
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
            
        print(f"CDC aplicado com sucesso em {tabela_destino}!")

print("\nPROCESSAMENTO DA CAMADA PRATA CONCLUÍDO COM SUCESSO PARA TODAS AS TABELAS!")

Tabelas encontradas na Bronze para processar: ['lnd_bronze_customers', 'lnd_bronze_geolocation', 'lnd_bronze_order_items', 'lnd_bronze_order_payments', 'lnd_bronze_order_reviews', 'lnd_bronze_orders', 'lnd_bronze_product_category_name_translation', 'lnd_bronze_products', 'lnd_bronze_sellers']


INICIANDO PIPELINE DA TABELA: lnd_bronze_customers
   Padronizado: lnd_bronze_customers colunas ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']...
Deduplicando os dados usando a chave primária: customer_id
Tabela silver_olist.stg_customers detectada. Executando MERGE incremental (CDC)...
CDC aplicado com sucesso em silver_olist.stg_customers!

INICIANDO PIPELINE DA TABELA: lnd_bronze_geolocation
   Padronizado: lnd_bronze_geolocation colunas ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']...
Deduplicando os dados usando a chave primária: geolocation_zip_code_prefix
Tabela silver_ol

In [0]:
# Banco Realizado na camada Bronze
banco_silver = "silver_olist"

#Lista dinamica de todas as tabelas da Bronze
tabelas_silver = spark.catalog.listTables(banco_silver)

#lista apenas com os nomes delas
lista_de_tabelas = [t.name for t in tabelas_silver if t.name.startswith("stg")]

print(lista_de_tabelas)

['stg_customers', 'stg_geolocation', 'stg_order_items', 'stg_order_payments', 'stg_order_reviews', 'stg_orders', 'stg_product_category_name_translation', 'stg_products', 'stg_sellers']


In [0]:
%sql
SELECT * 
FROM stg_customers

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,39664,mendonca,MG
00062b33cb9f6fe976afdcff967ea74d,f90f55ee274a4ae21510b386134b09cd,2306,sao paulo,SP
000bf8121c3412d3057d32371c5d3395,1bc9b2dad6aefbfbc011508e34c8adfc,12335,jacarei,SP
000f17e290c26b28549908a04cfe36c1,74541fbb7526dabecd0fc96b1192b9a7,98400,frederico westphalen,RS
000fd45d6fedae68fc6676036610f879,cee6fa72fb403ef9554b41581108084b,12970,piracaia,SP
001051abfcfdbed9f87b4266213a5df1,4ea5df5937187bd3176aee08d4782104,2251,sao paulo,SP
00114026c1b7b52ab1773f317ef4880b,f4dc0a81a11d3d270ccf5a9c4b5b187b,22470,rio de janeiro,RJ
0013cd8e350a7cc76873441e431dd5ee,334fed5abcee3aa96c13f1432703e1fd,3585,sao paulo,SP
0015f7887e2fde13ddaa7b8e385af919,866c923cde750dfc8cfbcf9d5ced0ee4,25903,mage,RJ
0018c09f333634ca9c80d9ff46e43e9c,688bb6776d5f885e4b46d4c4bac240c2,9190,santo andre,SP
